# 과제 10 · 10 · 두 담당이 동시에 근거를 모으고 합치기

## 현업 시나리오
근거는 두 곳에서 옵니다. 매뉴얼(검색)과 정비 이력(Tool Agent)입니다. 둘은 서로를 기다릴 이유가 없으므로 동시에 돌립니다. 그런데 둘이 같은 칸(evidence)에 결과를 쓰면 어느 쪽을 남길지 정해야 하고, 같은 근거를 둘 다 찾으면 한 번만 남겨야 합니다.

## 학습 목표
- START 에서 두 노드로 갈라 병렬로 돌린다.
- reducer 없이 같은 칸에 쓰면 그래프가 거부하는 것을 본다.
- `Annotated[list, operator.add]` 로 두 결과를 모두 남기고, 합치는 노드에서 중복을 지운다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · reducer 없이 같은 칸에 쓰면

In [ ]:
# 매뉴얼 담당과 이력 담당이 같은 칸(evidence)에 씁니다. 둘은 같은 단계에서 동시에 돕니다.
# 칸에 합치는 법(reducer)이 없으면 LangGraph 는 둘 중 무엇을 남길지 모릅니다.
# 관찰 포인트: 조용히 하나를 버리지 않고 InvalidUpdateError 로 거부합니다.
from typing import Annotated, TypedDict
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph

def practice_manual(state):
    return {"evidence": ["MC01-MM 4.2.2", "MC01-MM 4.2.3"], "trace": ["manual"]}

def practice_history(state):
    return {"evidence": ["MR-0102", "MC01-MM 4.2.2"], "trace": ["history"]}

def practice_build(state_type, merge=None):
    graph = StateGraph(state_type)
    graph.add_node("manual", practice_manual)
    graph.add_node("history", practice_history)
    graph.add_node("merge", merge or (lambda s: {}))
    graph.add_edge(START, "manual")          # 두 담당은 서로를 기다리지 않습니다
    graph.add_edge(START, "history")
    graph.add_edge("manual", "merge")
    graph.add_edge("history", "merge")
    graph.add_edge("merge", END)
    return graph.compile()

class PracticePlain(TypedDict, total=False):
    evidence: list
    trace: list

try:
    practice_build(PracticePlain).invoke({"trace": []})
    practice_refused = None
except InvalidUpdateError as error:
    practice_refused = type(error).__name__
print("reducer 없이:", practice_refused)

**결과 해설** — 두 담당이 같은 단계에서 같은 칸에 쓰자 그래프가 `InvalidUpdateError` 로 거부했습니다. 한쪽을 조용히 버렸다면 매뉴얼 근거나 이력 근거 중 하나가 소리 없이 사라졌을 것입니다.

### 2단계 · reducer 를 주면 둘 다 남는다

In [ ]:
# 칸의 타입에 합치는 법을 적습니다. operator.add 는 두 목록을 이어 붙입니다.
# trace 칸에도 같은 reducer 를 줘서 어느 담당이 돌았는지 남깁니다.
# 관찰 포인트: 두 담당의 결과가 모두 남습니다. 둘 다 찾은 4.2.2 는 두 번 들어 있습니다.
import operator

class PracticeMerged(TypedDict, total=False):
    evidence: Annotated[list, operator.add]
    trace: Annotated[list, operator.add]

practice_merged = practice_build(PracticeMerged).invoke({"trace": []})
print("근거:", practice_merged["evidence"])
print("돈 순서:", practice_merged["trace"])

**결과 해설** — 네 개가 모두 남았고, 두 담당 모두 merge 전에 돌았습니다. 대신 매뉴얼 담당과 이력 담당이 둘 다 찾은 4.2.2 가 두 번 들어 있습니다. 이것을 그대로 초안에 넘기면 같은 근거를 두 번 인용합니다.

### 3단계 · 합치는 노드에서 중복 지우기

In [ ]:
# merge 노드는 두 담당이 모두 끝난 뒤 한 번 돕니다. 여기서 같은 근거를 한 번만 남깁니다.
# 순서는 처음 나온 순서를 지킵니다. 결과는 새 칸(deduped)에 씁니다.
# 관찰 포인트: reducer 가 붙은 칸에 다시 쓰면 또 이어 붙습니다. 정리한 결과는 다른 칸에 둡니다.
class PracticeDeduped(PracticeMerged, total=False):
    deduped: list

def practice_merge(state):
    return {"deduped": list(dict.fromkeys(state["evidence"])), "trace": ["merge"]}

practice_deduped = practice_build(PracticeDeduped, practice_merge).invoke({"trace": []})
print("합친 근거:", practice_deduped["deduped"])
print("돈 순서:", practice_deduped["trace"])

assert practice_refused == "InvalidUpdateError", "reducer 없이도 같은 칸에 쓸 수 있었습니다"
assert sorted(practice_merged["evidence"]) == sorted(["MC01-MM 4.2.2", "MC01-MM 4.2.3", "MR-0102", "MC01-MM 4.2.2"])
assert set(practice_merged["trace"]) == {"manual", "history"}
assert sorted(practice_deduped["deduped"]) == ["MC01-MM 4.2.2", "MC01-MM 4.2.3", "MR-0102"]
assert practice_deduped["trace"][-1] == "merge"

**결과 해설** — 중복이 사라져 근거가 세 개가 되었고, merge 는 두 담당이 끝난 뒤 마지막에 한 번 돌았습니다. 앱의 merge 는 여기에 더해 판정 기준을 코드로 다시 계산합니다. 이력 담당 Agent 가 판정 결과를 어떻게 읽었든 판정은 코드가 정합니다.

## 직접 해 볼 과제
`practice_try_merge` 를 바꿔 이력 근거(MR-로 시작)를 매뉴얼 근거보다 앞에 두도록 정렬해 보세요. 실제 앱은 이력 근거를 먼저 둡니다. 근거 순서가 초안이나 보고서에 어떤 영향을 줄지 생각해 보세요.

In [ ]:
# merge 노드만 바꿔 같은 그래프를 다시 만듭니다.
# sorted 의 key 로 이력(MR-)을 앞에 둡니다. 같은 종류 안에서는 처음 나온 순서를 지킵니다.
def practice_try_merge(state):
    unique = list(dict.fromkeys(state["evidence"]))
    return {"deduped": sorted(unique, key=lambda i: not i.startswith("MR-")), "trace": ["merge"]}

print(practice_build(PracticeDeduped, practice_try_merge).invoke({"trace": []})["deduped"])

## 중간 결과
reducer 없이 돌렸을 때의 거부, reducer 를 줬을 때 두 담당의 결과가 모두 남는 것(중복 포함), 그리고 merge 노드가 중복을 지우고 마지막에 도는 것을 확인합니다.

## 실패 경계
병렬이라고 해서 빨라지는 것은 두 담당이 정말 서로를 기다리지 않을 때뿐입니다. 이력 담당 안에서 판정 → 이력 조회는 순서가 있으므로 그 안은 병렬로 바꿀 수 없습니다(Notebook 06). 또 두 담당 중 하나가 예외를 내면 그래프 전체가 멈춥니다. 앱은 설정이 빠진 경우(키, MCP 서버)를 503 과 이유로 돌려줍니다.

## 실제 app 연결
과제 10 App 의 `review.py::collect_manual` 과 `review.py::collect_history` 가 START 에서 함께 시작하고, 두 칸(`manual_found`, `history_found`)에 `operator.add` reducer 가 붙어 있습니다(`review.py::CaseState`). `review.py::merge` 가 같은 근거를 한 번만 남기고 판정 기준을 다시 계산합니다.

## 다음 Notebook 연결
다음 `11_review_pause.ipynb` 에서는 검사가 끝난 초안 앞에서 그래프를 멈추고 사람의 결정을 기다립니다.